# D&D 5E Monster HP Prediction Model

This notebook builds a machine learning model to predict **average Hit Points (HP)** for D&D 5E monsters using interpretable features.

## Goals
1. Predict HP based on **interpretable, actionable features** for monster design
2. Compare **Linear Regression** (interpretability) vs **Random Forest** (accuracy)
3. Identify which features contribute most to HP
4. Enable custom monster creation with clear parameter guidance

## Key Design Principles
- **Exclude ability scores**: No STR, DEX, CON, INT, WIS, CHA or modifiers
- **Exclude HP-derived features**: No hp_dice_count, hp_dice_size, effective_hp calculation
- **Include CR as predictor**: User knows target CR, wants appropriate HP
- **Prioritize interpretability**: Fewer features, clearer meaning
- **Use resistance_count**: Represents survivability/effective HP concept

## Approach
- **Two models**: Linear Regression (coefficients) + Random Forest (importance)
- **~51 features**: Core stats, action economy, special abilities, size/type
- **Interpretability analysis**: Plain English translations of coefficients
- **Practical guidance**: Marginal effects calculator for monster building

## 1. Setup & Data Loading

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
import pickle
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

In [ ]:
# Load the monster data
df = pd.read_csv('dnd5e_monsters_2014.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

## 2. Exploratory Data Analysis

In [ ]:
### Parse target variable and CR

def parse_hp_avg(hp_str):
    """Extract average HP from string like '135 (18d10+36)'"""
    if pd.isna(hp_str):
        return np.nan
    match = re.match(r'(\d+)', str(hp_str))
    return int(match.group(1)) if match else np.nan

def cr_to_numeric(cr_str):
    """Convert CR string to numeric value"""
    if pd.isna(cr_str):
        return np.nan
    
    cr_str = str(cr_str).strip()
    
    # Handle fractions
    if '/' in cr_str:
        num, denom = cr_str.split('/')
        return float(num) / float(denom)
    
    return float(cr_str)

# Apply parsers
df['hp_avg'] = df['HP'].apply(parse_hp_avg)
df['cr_numeric'] = df['Challenge_Rating'].apply(cr_to_numeric)

print("Sample parsed values:")
print(df[['Name', 'HP', 'hp_avg', 'Challenge_Rating', 'cr_numeric']].head(10))

In [ ]:
### Visualize HP distribution

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# HP distribution
axes[0, 0].hist(df['hp_avg'].dropna(), bins=40, edgecolor='black')
axes[0, 0].set_xlabel('Average HP')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('HP Distribution')

# HP vs CR scatter
axes[0, 1].scatter(df['cr_numeric'], df['hp_avg'], alpha=0.6)
axes[0, 1].set_xlabel('Challenge Rating')
axes[0, 1].set_ylabel('Average HP')
axes[0, 1].set_title('HP vs CR')
axes[0, 1].grid(True, alpha=0.3)

# Calculate correlation
corr = df[['cr_numeric', 'hp_avg']].corr().iloc[0, 1]
axes[0, 1].text(0.05, 0.95, f'Correlation: {corr:.3f}', 
                transform=axes[0, 1].transAxes, 
                verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# HP by CR tier (box plot)
df['cr_tier'] = pd.cut(df['cr_numeric'], 
                       bins=[0, 1, 5, 10, 16, 30], 
                       labels=['0-1', '2-5', '6-10', '11-16', '17+'])
df.boxplot(column='hp_avg', by='cr_tier', ax=axes[1, 0])
axes[1, 0].set_xlabel('CR Tier')
axes[1, 0].set_ylabel('Average HP')
axes[1, 0].set_title('HP Distribution by CR Tier')
plt.sca(axes[1, 0])
plt.xticks(rotation=0)

# Log scale HP vs CR
axes[1, 1].scatter(df['cr_numeric'], df['hp_avg'], alpha=0.6)
axes[1, 1].set_xlabel('Challenge Rating')
axes[1, 1].set_ylabel('Average HP (log scale)')
axes[1, 1].set_yscale('log')
axes[1, 1].set_title('HP vs CR (Log Scale)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nHP-CR Correlation: {corr:.4f}")
print("\n⭐ Key Insight: Strong HP-CR correlation (~0.95)")
print("   Justification: User knows target CR and wants appropriate HP")
print("   CR is included as a predictor for practical monster design.")

In [ ]:
### Check for missing values

print("Missing values in key columns:")
missing = df[['HP', 'Challenge_Rating', 'AC', 'Speed']].isnull().sum()
print(missing)

print(f"\nTotal monsters with valid HP and CR: {df[['hp_avg', 'cr_numeric']].notna().all(axis=1).sum()}")

## 3. Feature Engineering

We create ~51 interpretable features across 9 categories:
- Core (2): CR, AC
- Speed (7): Ground, fly, swim, burrow, climb, max, types count
- Saves/Skills (2): Proficiency counts
- Damage Coverage (4): **resistance_count (survivability proxy)**, immunity, vulnerability, condition immunity
- Senses (6): Darkvision, blindsight, truesight, tremorsense, passive perception
- Action Economy (8): Trait, action, reaction, bonus action, legendary action counts
- Attack Features (3): Multiattack, highest attack bonus, highest save DC
- Special Abilities (5): Legendary resistance, magic resistance, regeneration, spellcasting
- Size & Type (37): Size ordinal + type one-hot encoding

In [ ]:
### A. Core features: AC value

def parse_ac_value(ac_str):
    """Extract AC value from '17 (Natural Armor)' or '10'"""
    if pd.isna(ac_str):
        return np.nan
    match = re.match(r'(\d+)', str(ac_str))
    return int(match.group(1)) if match else np.nan

df['ac_value'] = df['AC'].apply(parse_ac_value)

print("Core features:")
print(df[['Name', 'cr_numeric', 'ac_value']].head())

In [ ]:
### B. Speed features

def parse_speed_features(speed_str):
    """Extract all speed types from '40 ft., fly 80 ft., swim 40 ft.'"""
    if pd.isna(speed_str):
        return pd.Series([0, 0, 0, 0, 0, 0, 0])
    
    speed_str = str(speed_str).lower()
    
    # Extract ground speed (first number)
    ground_match = re.match(r'(\d+)', speed_str)
    ground = int(ground_match.group(1)) if ground_match else 0
    
    # Extract other speeds
    fly = int(re.search(r'fly (\d+)', speed_str).group(1)) if re.search(r'fly (\d+)', speed_str) else 0
    swim = int(re.search(r'swim (\d+)', speed_str).group(1)) if re.search(r'swim (\d+)', speed_str) else 0
    burrow = int(re.search(r'burrow (\d+)', speed_str).group(1)) if re.search(r'burrow (\d+)', speed_str) else 0
    climb = int(re.search(r'climb (\d+)', speed_str).group(1)) if re.search(r'climb (\d+)', speed_str) else 0
    
    max_speed = max(ground, fly, swim, burrow, climb)
    movement_types = sum([1 for s in [ground, fly, swim, burrow, climb] if s > 0])
    
    return pd.Series([ground, fly, swim, burrow, climb, max_speed, movement_types])

df[['speed_ground', 'speed_fly', 'speed_swim', 'speed_burrow', 'speed_climb', 'max_speed', 'movement_types_count']] = \
    df['Speed'].apply(parse_speed_features)

print("Speed features:")
print(df[['Name', 'Speed', 'speed_ground', 'speed_fly', 'max_speed', 'movement_types_count']].head())

In [ ]:
### C. Saves and Skills

def count_saves(saves_str):
    """Count number of saving throw proficiencies"""
    if pd.isna(saves_str) or saves_str == '':
        return 0
    return len(re.findall(r'\w+ [+-]\d+', str(saves_str)))

def count_skills(skills_str):
    """Count number of skill proficiencies"""
    if pd.isna(skills_str) or skills_str == '':
        return 0
    return len(re.findall(r'\w+ [+-]\d+', str(skills_str)))

df['save_proficiency_count'] = df['Saving_Throws'].apply(count_saves)
df['skill_proficiency_count'] = df['Skills'].apply(count_skills)

print("Saves/Skills:")
print(df[['Name', 'save_proficiency_count', 'skill_proficiency_count']].head())

In [ ]:
### D. Damage Coverage (Survivability Metrics)

def count_items(text_str):
    """Count comma-separated items"""
    if pd.isna(text_str) or text_str == '':
        return 0
    items = re.split(r'[,;]', str(text_str))
    return len([item.strip() for item in items if item.strip()])

df['resistance_count'] = df['Resistances'].apply(count_items)
df['immunity_count'] = df['Immunities'].apply(count_items)
df['vulnerability_count'] = df['Vulnerabilities'].apply(count_items)
df['condition_immunity_count'] = df['Condition_Immunities'].apply(count_items)

print("Damage Coverage (Survivability):")
print(df[['Name', 'resistance_count', 'immunity_count', 'vulnerability_count', 'condition_immunity_count']].head(10))
print("\n⭐ Note: resistance_count serves as proxy for effective HP/survivability")

In [ ]:
### E. Senses features

def parse_senses(senses_str):
    """Extract vision types and ranges"""
    if pd.isna(senses_str):
        senses_str = ''
    senses_str = str(senses_str).lower()
    
    # Darkvision
    darkvision_match = re.search(r'darkvision (\d+)', senses_str)
    has_darkvision = 1 if darkvision_match else 0
    darkvision_range = int(darkvision_match.group(1)) if darkvision_match else 0
    
    # Other senses
    has_blindsight = 1 if 'blindsight' in senses_str else 0
    has_truesight = 1 if 'truesight' in senses_str else 0
    has_tremorsense = 1 if 'tremorsense' in senses_str else 0
    
    return pd.Series([has_darkvision, darkvision_range, has_blindsight, has_truesight, has_tremorsense])

df[['has_darkvision', 'darkvision_range', 'has_blindsight', 'has_truesight', 'has_tremorsense']] = \
    df['Senses'].apply(parse_senses)

df['passive_perception'] = pd.to_numeric(df['Passive_Perception'], errors='coerce').fillna(10)

print("Senses features:")
print(df[['Name', 'has_darkvision', 'darkvision_range', 'has_blindsight', 'passive_perception']].head())

In [ ]:
### F. Action Economy features

def count_abilities(ability_str):
    """Count number of abilities (traits/actions) separated by ' | '"""
    if pd.isna(ability_str) or ability_str == '':
        return 0
    return len([a for a in str(ability_str).split(' | ') if a.strip()])

df['trait_count'] = df['Traits'].apply(count_abilities)
df['action_count'] = df['Actions'].apply(count_abilities)
df['reaction_count'] = df['Reactions'].apply(count_abilities)
df['bonus_action_count'] = df['Bonus_Actions'].apply(count_abilities)

# Legendary actions
df['legendary_action_count'] = df['Legendary_Actions'].apply(count_abilities)
df['has_legendary_actions'] = (df['legendary_action_count'] > 0).astype(int)
df['legendary_actions_per_round'] = pd.to_numeric(df['Legendary_Actions_Num'], errors='coerce').fillna(0)

# Total abilities
df['total_ability_count'] = df['trait_count'] + df['action_count'] + df['reaction_count'] + \
                            df['bonus_action_count'] + df['legendary_action_count']

print("Action Economy:")
print(df[['Name', 'trait_count', 'action_count', 'legendary_action_count', 'has_legendary_actions', 'total_ability_count']].head())

In [ ]:
### G. Attack features

def parse_attack_features(actions_str):
    """Extract multiattack, highest attack bonus, and save DC"""
    if pd.isna(actions_str):
        actions_str = ''
    actions_str = str(actions_str).lower()
    
    # Multiattack
    has_multiattack = 1 if 'multiattack' in actions_str else 0
    
    # Extract attack bonuses
    attack_bonuses = re.findall(r'\+(\d+) to hit', actions_str)
    highest_attack_bonus = max([int(b) for b in attack_bonuses]) if attack_bonuses else 0
    
    # Extract save DCs
    save_dcs = re.findall(r'dc (\d+)', actions_str)
    highest_save_dc = max([int(dc) for dc in save_dcs]) if save_dcs else 0
    
    return pd.Series([has_multiattack, highest_attack_bonus, highest_save_dc])

df[['has_multiattack', 'highest_attack_bonus', 'highest_save_dc']] = \
    df['Actions'].apply(parse_attack_features)

print("Attack features:")
print(df[['Name', 'has_multiattack', 'highest_attack_bonus', 'highest_save_dc']].head())

In [ ]:
### H. Special ability features

# Combine all ability text for pattern matching
combined_abilities = (df['Traits'].fillna('') + ' ' + df['Actions'].fillna('') + ' ' + 
                     df['Reactions'].fillna('') + ' ' + df['Legendary_Actions'].fillna(''))

df['has_legendary_resistance'] = combined_abilities.str.contains('legendary resistance', case=False, na=False).astype(int)
df['has_magic_resistance'] = combined_abilities.str.contains('magic resistance', case=False, na=False).astype(int)
df['has_regeneration'] = combined_abilities.str.contains('regenerat', case=False, na=False).astype(int)
df['has_spellcasting'] = combined_abilities.str.contains('spellcasting', case=False, na=False).astype(int)

# Extract spellcaster level
def extract_spellcaster_level(text):
    if pd.isna(text):
        return 0
    match = re.search(r'(\d+)(?:st|nd|rd|th)-level spellcaster', str(text).lower())
    return int(match.group(1)) if match else 0

df['spellcaster_level'] = combined_abilities.apply(extract_spellcaster_level)

print("Special abilities:")
print(df[['Name', 'has_legendary_resistance', 'has_magic_resistance', 'has_regeneration', 
          'has_spellcasting', 'spellcaster_level']].head())

In [ ]:
### I. Size and Type encoding

# Size ordinal encoding
size_mapping = {
    'Tiny': 1,
    'Small': 2,
    'Medium': 3,
    'Large': 4,
    'Huge': 5,
    'Gargantuan': 6
}
df['size_ordinal'] = df['Size'].map(size_mapping).fillna(3)

# Type one-hot encoding
type_dummies = pd.get_dummies(df['Type'], prefix='type')
df = pd.concat([df, type_dummies], axis=1)

print(f"Size encoding: {df['size_ordinal'].value_counts().sort_index()}")
print(f"\nType columns created: {len(type_dummies.columns)} types")
print(f"Type columns: {type_dummies.columns.tolist()[:10]}...")

In [ ]:
### Summary of engineered features

feature_columns = [
    # Core (2)
    'cr_numeric', 'ac_value',
    # Speed (7)
    'speed_ground', 'speed_fly', 'speed_swim', 'speed_burrow', 'speed_climb', 
    'max_speed', 'movement_types_count',
    # Saves/Skills (2)
    'save_proficiency_count', 'skill_proficiency_count',
    # Damage Coverage (4) - Survivability Metrics
    'resistance_count', 'immunity_count', 'vulnerability_count', 'condition_immunity_count',
    # Senses (6)
    'has_darkvision', 'darkvision_range', 'has_blindsight', 'has_truesight', 
    'has_tremorsense', 'passive_perception',
    # Action Economy (8)
    'trait_count', 'action_count', 'reaction_count', 'bonus_action_count',
    'legendary_action_count', 'has_legendary_actions', 'legendary_actions_per_round',
    'total_ability_count',
    # Attack Features (3)
    'has_multiattack', 'highest_attack_bonus', 'highest_save_dc',
    # Special Abilities (5)
    'has_legendary_resistance', 'has_magic_resistance', 'has_regeneration',
    'has_spellcasting', 'spellcaster_level',
    # Size (1)
    'size_ordinal'
] + [col for col in df.columns if col.startswith('type_')]

print(f"Total engineered features: {len(feature_columns)}")
print(f"\nFeature breakdown:")
print(f"  Core: 2")
print(f"  Speed: 7")
print(f"  Saves/Skills: 2")
print(f"  Damage Coverage: 4 (resistance_count = survivability proxy)")
print(f"  Senses: 6")
print(f"  Action Economy: 8")
print(f"  Attack Features: 3")
print(f"  Special Abilities: 5")
print(f"  Size & Type: {1 + len([col for col in df.columns if col.startswith('type_')])}")
print(f"\n⭐ Excluded: All ability scores/modifiers, HP-derived features (hp_dice_count, hp_dice_size, effective_hp calculation)")

## 4. Model Training

In [ ]:
### Prepare data

X = df[feature_columns].copy()
y = df['hp_avg'].copy()

# Fill any remaining NaN values
X = X.fillna(0)

# Remove rows where HP is NaN
valid_idx = ~y.isna()
X = X[valid_idx]
y = y[valid_idx]

print(f"Final dataset shape: {X.shape}")
print(f"Features: {X.shape[1]}")
print(f"Samples: {X.shape[0]}")
print(f"\nTarget (HP) range: {y.min()} to {y.max()}")
print(f"Target (HP) mean: {y.mean():.1f}")
print(f"Target (HP) median: {y.median():.1f}")

# Train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nTrain set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

In [ ]:
### Train Linear Regression

# Scale features for linear regression (required for interpretable coefficients)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_lr_train = lr_model.predict(X_train_scaled)
y_pred_lr = lr_model.predict(X_test_scaled)

# Metrics
lr_mae_train = mean_absolute_error(y_train, y_pred_lr_train)
lr_mae = mean_absolute_error(y_test, y_pred_lr)
lr_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))
lr_r2_train = r2_score(y_train, y_pred_lr_train)
lr_r2 = r2_score(y_test, y_pred_lr)

print("Linear Regression Performance:")
print(f"  Train R²: {lr_r2_train:.4f}")
print(f"  Test R²: {lr_r2:.4f}")
print(f"  Train MAE: {lr_mae_train:.2f} HP")
print(f"  Test MAE: {lr_mae:.2f} HP")
print(f"  Test RMSE: {lr_rmse:.2f} HP")
print(f"\n⭐ Interpretability: Coefficients show HP change per unit increase in each feature")

In [ ]:
### Train Random Forest

# Train model (no scaling needed for tree-based models)
rf_model = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predictions
y_pred_rf_train = rf_model.predict(X_train)
y_pred_rf = rf_model.predict(X_test)

# Metrics
rf_mae_train = mean_absolute_error(y_train, y_pred_rf_train)
rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
rf_r2_train = r2_score(y_train, y_pred_rf_train)
rf_r2 = r2_score(y_test, y_pred_rf)

print("Random Forest Performance:")
print(f"  Train R²: {rf_r2_train:.4f}")
print(f"  Test R²: {rf_r2:.4f}")
print(f"  Train MAE: {rf_mae_train:.2f} HP")
print(f"  Test MAE: {rf_mae:.2f} HP")
print(f"  Test RMSE: {rf_rmse:.2f} HP")
print(f"\n⭐ Performance: Feature importance shows relative contribution to predictions")

In [ ]:
### Model Comparison

comparison_df = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'Train R²': [lr_r2_train, rf_r2_train],
    'Test R²': [lr_r2, rf_r2],
    'Train MAE': [lr_mae_train, rf_mae_train],
    'Test MAE': [lr_mae, rf_mae],
    'Test RMSE': [lr_rmse, rf_rmse]
})

print("\nModel Comparison:")
print(comparison_df.to_string(index=False))

# Determine success
print("\n=== Success Criteria ===")
print(f"Linear Regression R² > 0.85: {'✅ PASS' if lr_r2 > 0.85 else '❌ FAIL'} ({lr_r2:.4f})")
print(f"Random Forest R² > 0.90: {'✅ PASS' if rf_r2 > 0.90 else '❌ FAIL'} ({rf_r2:.4f})")

In [ ]:
### Visualize: Predicted vs Actual

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Linear Regression
axes[0].scatter(y_test, y_pred_lr, alpha=0.6, edgecolors='k', linewidth=0.5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
             'r--', lw=2, label='Perfect prediction')
axes[0].set_xlabel('Actual HP', fontsize=12)
axes[0].set_ylabel('Predicted HP', fontsize=12)
axes[0].set_title(f'Linear Regression: Predicted vs Actual\nR² = {lr_r2:.4f}, MAE = {lr_mae:.2f}', fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Random Forest
axes[1].scatter(y_test, y_pred_rf, alpha=0.6, edgecolors='k', linewidth=0.5)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
             'r--', lw=2, label='Perfect prediction')
axes[1].set_xlabel('Actual HP', fontsize=12)
axes[1].set_ylabel('Predicted HP', fontsize=12)
axes[1].set_title(f'Random Forest: Predicted vs Actual\nR² = {rf_r2:.4f}, MAE = {rf_mae:.2f}', fontsize=13)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
### Visualize: Residuals

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Linear Regression residuals
lr_residuals = y_test - y_pred_lr
axes[0].scatter(y_pred_lr, lr_residuals, alpha=0.6, edgecolors='k', linewidth=0.5)
axes[0].axhline(y=0, color='r', linestyle='--', lw=2)
axes[0].set_xlabel('Predicted HP', fontsize=12)
axes[0].set_ylabel('Residual (Actual - Predicted)', fontsize=12)
axes[0].set_title('Linear Regression: Residuals Plot', fontsize=13)
axes[0].grid(True, alpha=0.3)

# Random Forest residuals
rf_residuals = y_test - y_pred_rf
axes[1].scatter(y_pred_rf, rf_residuals, alpha=0.6, edgecolors='k', linewidth=0.5)
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted HP', fontsize=12)
axes[1].set_ylabel('Residual (Actual - Predicted)', fontsize=12)
axes[1].set_title('Random Forest: Residuals Plot', fontsize=13)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
### Accuracy by CR Tier

# Add CR tier to test set
test_indices = X_test.index
cr_tiers_test = df.loc[test_indices, 'cr_tier']

# Calculate MAE by tier
tier_performance = pd.DataFrame({
    'CR Tier': cr_tiers_test,
    'LR Residual': abs(lr_residuals),
    'RF Residual': abs(rf_residuals)
}).groupby('CR Tier').mean()

print("Mean Absolute Error by CR Tier:")
print(tier_performance)

# Visualize
tier_performance.plot(kind='bar', figsize=(10, 6))
plt.xlabel('CR Tier')
plt.ylabel('Mean Absolute Error (HP)')
plt.title('Model Accuracy by CR Tier')
plt.legend(['Linear Regression', 'Random Forest'])
plt.xticks(rotation=0)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 5. Interpretability Analysis ⭐ CORE VALUE

In [ ]:
### Linear Regression Coefficients

# Create coefficients DataFrame
lr_coefficients = pd.DataFrame({
    'feature': feature_columns,
    'coefficient': lr_model.coef_,
    'abs_coefficient': np.abs(lr_model.coef_)
}).sort_values('abs_coefficient', ascending=False)

print("Top 20 Features by Absolute Coefficient (Linear Regression):")
print(lr_coefficients.head(20).to_string(index=False))

# Visualize top 20
plt.figure(figsize=(12, 8))
top_20_coef = lr_coefficients.head(20)
colors = ['blue' if x > 0 else 'red' for x in top_20_coef['coefficient']]
plt.barh(range(len(top_20_coef)), top_20_coef['coefficient'], color=colors, alpha=0.7, edgecolor='black')
plt.yticks(range(len(top_20_coef)), top_20_coef['feature'])
plt.xlabel('Coefficient (HP change per standardized unit)', fontsize=12)
plt.title('Top 20 Linear Regression Coefficients\n(Blue = Positive impact, Red = Negative impact)', fontsize=13)
plt.axvline(x=0, color='black', linestyle='--', linewidth=1)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
### Plain English Translation of Coefficients

# Get feature std deviations for unscaling
feature_stds = pd.Series(scaler.scale_, index=feature_columns)

# Calculate real-world impact (per 1 unit change)
lr_coefficients['per_unit_impact'] = lr_coefficients.apply(
    lambda row: row['coefficient'] / feature_stds[row['feature']], axis=1
)

print("=== Plain English Translation ===")
print("\nTop 10 Features - HP Impact per Unit Increase:\n")

for idx, row in lr_coefficients.head(10).iterrows():
    feature = row['feature']
    impact = row['per_unit_impact']
    direction = "increases" if impact > 0 else "decreases"
    
    # Special formatting for key features
    if feature == 'cr_numeric':
        print(f"  CR: Each +1 CR {direction} HP by ~{abs(impact):.1f}")
    elif feature == 'ac_value':
        print(f"  AC: Each +1 AC {direction} HP by ~{abs(impact):.1f}")
    elif feature == 'size_ordinal':
        print(f"  Size: Each size category {direction} HP by ~{abs(impact):.1f}")
    elif feature == 'resistance_count':
        print(f"  Resistance Count (Survivability): Each +1 resistance {direction} HP by ~{abs(impact):.1f}")
    else:
        print(f"  {feature}: +1 {direction} HP by ~{abs(impact):.1f}")

In [ ]:
### Random Forest Feature Importance

rf_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 20 Features by Importance (Random Forest):")
print(rf_importance.head(20).to_string(index=False))

# Visualize top 20
plt.figure(figsize=(12, 8))
top_20_imp = rf_importance.head(20)
plt.barh(range(len(top_20_imp)), top_20_imp['importance'], alpha=0.7, edgecolor='black')
plt.yticks(range(len(top_20_imp)), top_20_imp['feature'])
plt.xlabel('Feature Importance', fontsize=12)
plt.title('Top 20 Random Forest Feature Importances', fontsize=13)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
### Model Agreement Analysis

# Merge LR coefficients and RF importances
agreement = lr_coefficients.merge(rf_importance, on='feature', suffixes=('_lr', '_rf'))

# Rank features by both models
agreement['lr_rank'] = agreement['abs_coefficient'].rank(ascending=False)
agreement['rf_rank'] = agreement['importance'].rank(ascending=False)
agreement['rank_diff'] = abs(agreement['lr_rank'] - agreement['rf_rank'])

# Features both models agree on (small rank difference)
print("=== High Agreement (Both Models Rank Highly) ===")
print("These features have strong linear and non-linear relationships:\n")
high_agreement = agreement[(agreement['lr_rank'] <= 20) & (agreement['rf_rank'] <= 20)].sort_values('rank_diff')
print(high_agreement[['feature', 'lr_rank', 'rf_rank', 'rank_diff']].head(10).to_string(index=False))

# Features with disagreement (large rank difference)
print("\n=== High Disagreement (Non-linear Relationships) ===")
print("Random Forest captures these better (non-linear patterns):\n")
high_disagreement = agreement.sort_values('rank_diff', ascending=False).head(10)
print(high_disagreement[['feature', 'lr_rank', 'rf_rank', 'rank_diff']].to_string(index=False))

In [ ]:
### CR-Adjusted Analysis (Supplementary)

# Train models WITHOUT CR to show what matters controlling for CR
feature_columns_no_cr = [f for f in feature_columns if f != 'cr_numeric']

X_no_cr = X[feature_columns_no_cr]
X_train_no_cr, X_test_no_cr, y_train_no_cr, y_test_no_cr = train_test_split(
    X_no_cr, y, test_size=0.2, random_state=42
)

# Linear Regression without CR
scaler_no_cr = StandardScaler()
X_train_no_cr_scaled = scaler_no_cr.fit_transform(X_train_no_cr)
X_test_no_cr_scaled = scaler_no_cr.transform(X_test_no_cr)

lr_model_no_cr = LinearRegression()
lr_model_no_cr.fit(X_train_no_cr_scaled, y_train_no_cr)
y_pred_lr_no_cr = lr_model_no_cr.predict(X_test_no_cr_scaled)
lr_r2_no_cr = r2_score(y_test_no_cr, y_pred_lr_no_cr)
lr_mae_no_cr = mean_absolute_error(y_test_no_cr, y_pred_lr_no_cr)

# Random Forest without CR
rf_model_no_cr = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf_model_no_cr.fit(X_train_no_cr, y_train_no_cr)
y_pred_rf_no_cr = rf_model_no_cr.predict(X_test_no_cr)
rf_r2_no_cr = r2_score(y_test_no_cr, y_pred_rf_no_cr)
rf_mae_no_cr = mean_absolute_error(y_test_no_cr, y_pred_rf_no_cr)

print("=== Models WITHOUT CR (Controlling for CR) ===")
print(f"\nLinear Regression: R² = {lr_r2_no_cr:.4f}, MAE = {lr_mae_no_cr:.2f} HP")
print(f"Random Forest: R² = {rf_r2_no_cr:.4f}, MAE = {rf_mae_no_cr:.2f} HP")
print(f"\nR² Drop (LR): {lr_r2 - lr_r2_no_cr:.4f}")
print(f"R² Drop (RF): {rf_r2 - rf_r2_no_cr:.4f}")
print("\n⭐ Insight: This shows what matters WITHIN a CR tier (for varying HP at same CR)")

# Top features without CR
rf_importance_no_cr = pd.DataFrame({
    'feature': feature_columns_no_cr,
    'importance': rf_model_no_cr.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Non-CR Features:")
print(rf_importance_no_cr.head(10).to_string(index=False))

## 6. Prediction Pipeline

In [ ]:
### Prediction Function

def predict_hp(monster_features, model='both'):
    """
    Predict HP for a custom monster.
    
    Parameters:
    -----------
    monster_features : dict or pd.Series
        Dictionary or Series with feature values
    model : str
        'lr' (Linear Regression), 'rf' (Random Forest), or 'both' (default)
    
    Returns:
    --------
    dict : Predicted HP values from selected model(s)
    """
    # Convert to DataFrame
    if isinstance(monster_features, dict):
        monster_df = pd.DataFrame([monster_features])
    elif isinstance(monster_features, pd.Series):
        monster_df = monster_features.to_frame().T
    else:
        monster_df = monster_features.copy()
    
    # Ensure all features are present
    for col in feature_columns:
        if col not in monster_df.columns:
            monster_df[col] = 0
    
    monster_df = monster_df[feature_columns]
    
    results = {}
    
    # Linear Regression prediction
    if model in ['lr', 'both']:
        monster_scaled = scaler.transform(monster_df)
        lr_pred = lr_model.predict(monster_scaled)[0]
        results['Linear Regression'] = max(1, lr_pred)  # HP can't be < 1
    
    # Random Forest prediction
    if model in ['rf', 'both']:
        rf_pred = rf_model.predict(monster_df)[0]
        results['Random Forest'] = max(1, rf_pred)
    
    # Average if both
    if model == 'both':
        results['Average'] = (results['Linear Regression'] + results['Random Forest']) / 2
    
    return results

print("Prediction function ready!")
print("\nUsage: predict_hp(monster_features, model='both')")
print("  model='lr' for Linear Regression only")
print("  model='rf' for Random Forest only")
print("  model='both' for both models + average")

In [ ]:
### Example Prediction 1: Validate with Goblin

goblin_idx = df[df['Name'] == 'Goblin'].index[0]
goblin_features = X.loc[goblin_idx]
goblin_actual_hp = y.loc[goblin_idx]
goblin_cr = df.loc[goblin_idx, 'cr_numeric']

print("Example 1: Goblin (Validation)")
print(f"  Actual HP: {goblin_actual_hp}")
print(f"  CR: {goblin_cr}")
print(f"\nPredictions:")
goblin_pred = predict_hp(goblin_features)
for model, hp in goblin_pred.items():
    error = hp - goblin_actual_hp
    print(f"  {model}: {hp:.1f} HP (error: {error:+.1f})")

In [ ]:
### Example Prediction 2: Scaled Monster ("What if Goblin was CR 5?")

# Create a hypothetical CR 5 goblin by scaling its features
scaled_goblin = goblin_features.copy()
scaled_goblin['cr_numeric'] = 5
scaled_goblin['ac_value'] = 15  # Higher AC for CR 5
scaled_goblin['has_multiattack'] = 1  # Add multiattack
scaled_goblin['highest_attack_bonus'] = 7  # Better attack bonus

print("\nExample 2: CR 5 'Goblin' (Scaled)")
print(f"  Original Goblin HP: {goblin_actual_hp} (CR {goblin_cr})")
print(f"  Modified features: CR 5, AC 15, Multiattack, +7 attack bonus")
print(f"\nPredicted HP for CR 5 version:")
scaled_pred = predict_hp(scaled_goblin)
for model, hp in scaled_pred.items():
    print(f"  {model}: {hp:.1f} HP")

In [ ]:
### Example Prediction 3: Custom Monster from Scratch

# Create a custom CR 10 monster
custom_monster = {
    'cr_numeric': 10,
    'ac_value': 17,
    'size_ordinal': 4,  # Large
    'speed_ground': 40,
    'speed_fly': 60,
    'max_speed': 60,
    'movement_types_count': 2,
    'has_multiattack': 1,
    'highest_attack_bonus': 9,
    'highest_save_dc': 16,
    'resistance_count': 2,  # Survivability metric
    'immunity_count': 1,
    'has_legendary_actions': 1,
    'legendary_action_count': 3,
    'legendary_actions_per_round': 3,
    'trait_count': 2,
    'action_count': 4,
    'has_darkvision': 1,
    'darkvision_range': 120,
    'passive_perception': 18,
    'save_proficiency_count': 3
}

print("\nExample 3: Custom CR 10 Monster")
print("  Features: Large, AC 17, fly 60ft, multiattack, legendary actions")
print("  Resistances: 2 (survivability), Immunities: 1")
print(f"\nPredicted HP:")
custom_pred = predict_hp(custom_monster)
for model, hp in custom_pred.items():
    print(f"  {model}: {hp:.1f} HP")

In [ ]:
### HP by CR Lookup Table

# Generate baseline features for each CR
cr_values = [0, 0.125, 0.25, 0.5, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 30]

cr_lookup = []
for cr in cr_values:
    # Create baseline monster at this CR
    baseline = {feat: 0 for feat in feature_columns}
    baseline['cr_numeric'] = cr
    baseline['ac_value'] = 10 + int(cr / 2)  # AC scales with CR
    baseline['size_ordinal'] = 3  # Medium by default
    baseline['speed_ground'] = 30
    baseline['max_speed'] = 30
    baseline['movement_types_count'] = 1
    
    # Add features based on CR
    if cr >= 2:
        baseline['has_multiattack'] = 1
    if cr >= 5:
        baseline['resistance_count'] = 1
    if cr >= 10:
        baseline['has_legendary_actions'] = 1
        baseline['legendary_action_count'] = 3
        baseline['legendary_actions_per_round'] = 3
    if cr >= 15:
        baseline['immunity_count'] = 1
        baseline['has_legendary_resistance'] = 1
    
    pred = predict_hp(baseline, model='rf')  # Use RF for lookup table
    hp = pred['Random Forest']
    
    cr_lookup.append({
        'CR': cr,
        'Predicted HP': int(hp),
        'Range (80-120%)': f"{int(hp * 0.8)}-{int(hp * 1.2)}"
    })

cr_lookup_df = pd.DataFrame(cr_lookup)
print("HP by CR Lookup Table (Baseline Monster):")
print(cr_lookup_df.to_string(index=False))

## 7. Practical Monster Building Guide

In [ ]:
### Marginal Effects Calculator

def marginal_effects_calculator(base_cr=10, adjustments=None):
    """
    Show how feature changes affect HP at a given CR.
    
    Parameters:
    -----------
    base_cr : float
        Target CR for the monster
    adjustments : dict
        Feature adjustments to show (e.g., {'size_ordinal': +1, 'ac_value': +2})
    """
    # Create baseline monster
    baseline = {feat: 0 for feat in feature_columns}
    baseline['cr_numeric'] = base_cr
    baseline['ac_value'] = 10 + int(base_cr / 2)
    baseline['size_ordinal'] = 3
    baseline['speed_ground'] = 30
    baseline['max_speed'] = 30
    baseline['movement_types_count'] = 1
    
    if base_cr >= 2:
        baseline['has_multiattack'] = 1
    if base_cr >= 10:
        baseline['has_legendary_actions'] = 1
        baseline['legendary_action_count'] = 3
    
    baseline_hp = predict_hp(baseline, model='rf')['Random Forest']
    
    print(f"=== HP Adjustments for CR {base_cr} Monster ===")
    print(f"Baseline HP: {baseline_hp:.1f}\n")
    print("Adjustments:")
    
    if adjustments:
        for feature, change in adjustments.items():
            adjusted = baseline.copy()
            adjusted[feature] = baseline.get(feature, 0) + change
            adjusted_hp = predict_hp(adjusted, model='rf')['Random Forest']
            hp_change = adjusted_hp - baseline_hp
            
            # Format feature name
            if feature == 'size_ordinal':
                size_names = {1: 'Tiny', 2: 'Small', 3: 'Medium', 4: 'Large', 5: 'Huge', 6: 'Gargantuan'}
                old_size = size_names.get(baseline.get(feature, 3), 'Medium')
                new_size = size_names.get(adjusted[feature], 'Medium')
                print(f"  size_ordinal {change:+d} ({old_size} → {new_size}): {hp_change:+.1f} HP")
            elif feature == 'ac_value':
                old_ac = baseline.get(feature, 0)
                new_ac = adjusted[feature]
                print(f"  ac_value {change:+d} (AC {old_ac} → {new_ac}): {hp_change:+.1f} HP")
            else:
                print(f"  {feature} {change:+d}: {hp_change:+.1f} HP")
    else:
        # Show default adjustments
        default_adjustments = {
            'size_ordinal': 1,
            'ac_value': 2,
            'has_legendary_actions': 1,
            'resistance_count': 2
        }
        marginal_effects_calculator(base_cr, default_adjustments)

# Example usage
marginal_effects_calculator(base_cr=10, adjustments={
    'size_ordinal': 1,  # Large → Huge
    'ac_value': 2,  # +2 AC
    'has_legendary_actions': 1,  # Add legendary actions
    'has_legendary_resistance': 1,  # Add legendary resistance
    'resistance_count': 2  # Add 2 resistances (survivability)
})

In [ ]:
### Design Recommendations

print("=== Monster Design Recommendations ===")
print("\n1. CR-HP Relationship Guidelines:")
print("   - Low CR (0-1): HP ~5-25, linear scaling")
print("   - Mid CR (2-10): HP ~20-150, exponential growth begins")
print("   - High CR (11-20): HP ~150-400, legendary features dominate")
print("   - Epic CR (21+): HP ~400+, multiple immunities/resistances expected")

print("\n2. Size Adjustment Rules:")
size_hp_impact = {}
for size_change in [1, 2, -1]:
    baseline = {'cr_numeric': 5, 'ac_value': 15, 'size_ordinal': 3}
    baseline = {**{feat: 0 for feat in feature_columns}, **baseline}
    adjusted = baseline.copy()
    adjusted['size_ordinal'] = baseline['size_ordinal'] + size_change
    
    baseline_hp = predict_hp(baseline, model='rf')['Random Forest']
    adjusted_hp = predict_hp(adjusted, model='rf')['Random Forest']
    
    size_hp_impact[size_change] = adjusted_hp - baseline_hp

print(f"   Medium → Large: +{size_hp_impact[1]:.1f} HP (CR 5 baseline)")
print(f"   Medium → Huge: +{size_hp_impact[2]:.1f} HP (CR 5 baseline)")
print(f"   Medium → Small: {size_hp_impact[-1]:.1f} HP (CR 5 baseline)")

print("\n3. AC vs HP Tradeoff Patterns:")
print("   - Higher AC monsters can have slightly lower HP")
print("   - General rule: +2 AC ≈ -5 to -15 HP (CR dependent)")
print("   - Tank monsters: High AC + High HP + Resistances")
print("   - Glass cannons: Low AC + Low HP + High damage")

print("\n4. Legendary Feature HP Correlations:")
legendary_features = ['has_legendary_actions', 'has_legendary_resistance', 'has_magic_resistance']
for feat in legendary_features:
    baseline = {'cr_numeric': 10, 'ac_value': 17, 'size_ordinal': 4}
    baseline = {**{f: 0 for f in feature_columns}, **baseline}
    with_feature = baseline.copy()
    with_feature[feat] = 1
    
    baseline_hp = predict_hp(baseline, model='rf')['Random Forest']
    with_feature_hp = predict_hp(with_feature, model='rf')['Random Forest']
    impact = with_feature_hp - baseline_hp
    
    print(f"   {feat}: {impact:+.1f} HP (CR 10)")

print("\n5. Resistance Count (Survivability):")
print("   - Acts as effective HP multiplier")
print("   - Each resistance adds survivability equivalent to ~5-20 HP (CR dependent)")
print("   - High CR monsters (15+) should have 2-4+ resistances/immunities")

## 8. Conclusions & Model Saving

In [ ]:
### Summary

print("=== HP Prediction Model Summary ===")
print("\n1. Model Performance:")
print(f"   Linear Regression: R² = {lr_r2:.4f}, MAE = {lr_mae:.2f} HP")
print(f"   Random Forest: R² = {rf_r2:.4f}, MAE = {rf_mae:.2f} HP")
print(f"\n   {'✅' if lr_r2 > 0.85 else '❌'} Linear Regression meets target (R² > 0.85)")
print(f"   {'✅' if rf_r2 > 0.90 else '❌'} Random Forest meets target (R² > 0.90)")

print("\n2. Key Findings:")
print(f"   - CR is the dominant predictor (r = {df[['cr_numeric', 'hp_avg']].corr().iloc[0, 1]:.3f})")
print(f"   - Top non-CR features (RF): {', '.join(rf_importance[rf_importance['feature'] != 'cr_numeric'].head(5)['feature'].tolist())}")
print(f"   - Resistance count effectively represents survivability/effective HP")
print(f"   - Size has moderate impact on HP (~{abs(size_hp_impact[1]):.0f} HP per category)")

print("\n3. Interpretability vs Accuracy:")
print("   - Linear Regression: Clear coefficients, slightly lower accuracy")
print("   - Random Forest: Higher accuracy, feature importance rankings")
print("   - Recommendation: Use Linear for understanding, Random Forest for predictions")

print("\n4. Practical Applications:")
print("   ✅ Predict HP for custom monsters")
print("   ✅ Understand feature impact on HP")
print("   ✅ Balance monsters by adjusting parameters")
print("   ✅ Identify HP outliers in existing monsters")

In [ ]:
### Limitations

print("=== Model Limitations ===")
print("\n1. HP-CR Circular Dependency:")
print("   - CR strongly correlates with HP in official monsters")
print("   - Model assumes user knows target CR (practical for design)")
print("   - May not generalize to non-standard CR/HP ratios")

print("\n2. Small Dataset:")
print(f"   - Only {X.shape[0]} monsters in training data")
print("   - Limited examples at very high CR (20+)")
print("   - Type imbalance (humanoids vs. other types)")

print("\n3. No Interaction Terms:")
print("   - Features treated independently in Linear Regression")
print("   - Synergies not explicitly modeled (e.g., flight + ranged attacks)")
print("   - Random Forest captures some interactions implicitly")

print("\n4. Missing Ability Scores:")
print("   - Excluded by design for interpretability")
print("   - CON modifier directly affects HP in D&D rules")
print("   - Trade-off: simpler model, slightly lower accuracy")

In [ ]:
### Usage Guidance

print("=== Usage Guidance ===")
print("\n1. For New Monsters:")
print("   - Start with CR lookup table for baseline HP")
print("   - Adjust based on size, defenses, special abilities")
print("   - Use marginal effects calculator to fine-tune")

print("\n2. For Balancing:")
print("   - Input custom stats to predict_hp()")
print("   - Compare predicted vs. desired HP")
print("   - Adjust features to hit target HP range")

print("\n3. For Variants:")
print("   - Take existing monster features")
print("   - Modify 1-2 parameters (e.g., add flight, increase size)")
print("   - Predict new HP based on changes")

print("\n4. For Analysis:")
print("   - Identify outliers: monsters with unusual HP for their CR")
print("   - Understand what makes high-HP monsters tanky")
print("   - Compare official monsters to your custom designs")

In [ ]:
### Save Models

# Save Linear Regression model and scaler
with open('hp_lr_model.pkl', 'wb') as f:
    pickle.dump(lr_model, f)

with open('hp_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save Random Forest model
with open('hp_rf_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

# Save feature columns list
with open('hp_feature_columns.pkl', 'wb') as f:
    pickle.dump(feature_columns, f)

print("Models saved successfully!")
print("\nFiles created:")
print("  - hp_lr_model.pkl (Linear Regression model)")
print("  - hp_rf_model.pkl (Random Forest model)")
print("  - hp_scaler.pkl (StandardScaler for Linear Regression)")
print("  - hp_feature_columns.pkl (Feature list)")
print("\nTo load and use:")
print("""\nimport pickle
with open('hp_rf_model.pkl', 'rb') as f:
    model = pickle.load(f)
with open('hp_feature_columns.pkl', 'rb') as f:
    features = pickle.load(f)
""")

---

## Next Steps

Potential enhancements:
1. **Confidence Intervals**: Add prediction uncertainty ranges
2. **Interaction Terms**: Explore polynomial/interaction features ("Advanced" section)
3. **Inverse Model**: Given target HP → suggest CR/features
4. **Interactive Widget**: Build Jupyter widget for real-time predictions
5. **Outlier Analysis**: Identify monsters with unusual HP for their CR
6. **Multioutput Model**: Predict both HP and CR simultaneously

**Happy Monster Building!** 🎲✨